# 04. Model Evaluation
This notebook evaluates the performance of the trained models using standardized metrics and generates figures for the final report.

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import mean_squared_error, r2_score
from src.visualization.plots import set_style, plot_actual_vs_predicted, plot_model_comparison
from src.config import PathConfig, ModelConfig

set_style()
print("🚀 Initializing Model Evaluation...")

try:
    # 1. Load Processed Data
    data = pd.read_csv(PathConfig.PROCESSED_DATA)
    X = data.drop(columns=[ModelConfig.TARGET_COL])
    y_true_log = np.log1p(data[ModelConfig.TARGET_COL])
    
    # 2. Evaluate XGBoost (Standard Deliverable)
    model_path = os.path.join(PathConfig.MODELS_DIR, 'model.pkl')
    results = {}
    
    if os.path.exists(model_path):
        model = joblib.load(model_path)
        y_pred_log = model.predict(X)
        
        rmse = np.sqrt(mean_squared_error(y_true_log, y_pred_log))
        r2 = r2_score(y_true_log, y_pred_log)
        results['XGBoost'] = rmse
        
        print(f"\n📈 XGBoost Evaluation Result:")
        print(f"   - RMSE (Log): {rmse:.4f}")
        print(f"   - R2 Score: {r2:.4f}")
        
        # Save Figures
        plot_actual_vs_predicted(
            y_true_log, y_pred_log, 
            title="XGBoost Performance: Actual vs Predicted", 
            save_path=os.path.join(PathConfig.REPORTS_DIR, 'figures/xgb_eval.png')
        )
    
    # 3. Model Comparison Plot
    # Note: Baseline is assumed to be trained and saved as baseline_lr.joblib
    baseline_path = os.path.join(PathConfig.MODELS_DIR, 'baseline_lr.joblib')
    if os.path.exists(baseline_path):
        baseline = joblib.load(baseline_path)
        y_base_log = baseline.predict(X)
        results['Baseline'] = np.sqrt(mean_squared_error(y_true_log, y_base_log))
        
    if results:
        plot_model_comparison(
            results, 
            metric_name="RMSE (Log Scale)", 
            save_path=os.path.join(PathConfig.REPORTS_DIR, 'figures/model_comparison.png')
        )
        print(f"\n✅ Evaluation figures saved to {os.path.join(PathConfig.REPORTS_DIR, 'figures/')}")
        
except Exception as e:
    print(f"❌ Error during evaluation: {e}")